In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification, TrainingArguments, Trainer
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
import librosa
import librosa.display
import matplotlib.pyplot as plt
import IPython.display as ipd
import random

### Basic EDA

In [ ]:
train_df = pd.read_csv('/kaggle/input/comsys-hackathon-6-task-b-2-0/train_new.csv') # columns: filename, target
test_df = pd.read_csv('/kaggle/input/comsys-hackathon-6-task-b-2-0/test_new_final.csv') # same

# Paths
test_audio_dir = '/kaggle/input/comsys-hackathon-6-task-b-2-0/ComsysTaskB/test'
train_audio_dir = '/kaggle/input/comsys-hackathon-6-task-b-2-0/ComsysTaskB/train'

train_df['file_path'] = train_df['filename'].apply(lambda x: os.path.join(train_audio_dir, x))
test_df['file_path'] = test_df['filename'].apply(lambda x: os.path.join(test_audio_dir, x))

print(train_df.head())
print(f"Train samples: {len(train_df)}, Test samples: {len(test_df)}")

In [ ]:
print("Unique targets in train:", sorted(train_df['target'].unique()))

In [ ]:
# Class distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=train_df, x='target')
plt.title('Class Distribution (0=real_female, 1=real_male, 2=fake_female, 3=fake_male)')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

# Real vs Fake
train_df['is_real'] = train_df['target'].isin([0, 1]).astype(int)
sns.countplot(data=train_df, x='is_real')
plt.title('Real (0) vs Fake (1)')
plt.show()

# Duration analysis
def get_duration(file_path, sr=16000):
    try:
        y, _ = librosa.load(file_path, sr=sr)
        return len(y) / sr
    except:
        return 0

train_df['duration'] = train_df['file_path'].apply(get_duration)
sns.boxplot(data=train_df, x='target', y='duration')
plt.title('Audio Duration by Class')
plt.show()

print(train_df.groupby('target')['duration'].describe())

### Sanity check for labels

In [ ]:
LABEL_MAP = {
    0: "Real Female",
    1: "Real Male",
    2: "Fake Female",
    3: "Fake Male"
}

df = pd.read_csv('/kaggle/input/comsys-hackathon-6-task-b-2-0/train_new.csv')
df['file_path'] = df['filename'].apply(lambda x: os.path.join('/kaggle/input/comsys-hackathon-6-task-b-2-0/ComsysTaskB/train', x))
df['class_name'] = df['target'].map(LABEL_MAP)

print("Class Distribution:")
print(df['class_name'].value_counts())
print("\nSample rows:")
print(df[['filename', 'target', 'class_name']].head())

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
fig.suptitle("Random Sample from Each Class", fontsize=16, fontweight='bold')

for idx, (target, class_name) in enumerate(LABEL_MAP.items()):
    class_files = df[df['target'] == target]['file_path'].tolist()
    if not class_files:
        print(f"No files found for class {target} ({class_name})")
        continue

    random_file = random.choice(class_files)
    y, sr = librosa.load(random_file, sr=16000)

    ax_wave = axes[idx, 0]
    librosa.display.waveshow(y, sr=sr, ax=ax_wave, color='darkcyan')
    ax_wave.set_title(f"{class_name}\nWaveform", fontsize=12)
    ax_wave.set_xlabel("Time (s)")

    ax_spec = axes[idx, 1]
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    img = librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', ax=ax_spec, cmap='magma')
    ax_spec.set_title(f"Mel Spectrogram", fontsize=12)

    print(f"\nPlaying: {os.path.basename(random_file)} → {class_name}")
    display(ipd.Audio(y, rate=sr))

plt.tight_layout()
plt.show()

## Classification Datset creation

In [ ]:
from transformers import Trainer, TrainingArguments , TrainerCallback

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification, Trainer, TrainingArguments, TrainerCallback
from torch.utils.data import Dataset
import torch
import numpy as np
import librosa
from sklearn.metrics import accuracy_score

train_df = train_df.rename(columns={"label": "target"})

# Use correct model
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=4,
    problem_type="single_label_classification"
)

# Move to GPU
if torch.cuda.is_available():
    model = model.cuda()

label2id = {0: 0, 1: 1, 2: 2, 3: 3}
id2label = {v: k for k, v in label2id.items()}

class AudioDataset(Dataset):
    def __init__(self, dataframe, processor, max_length=16000 * 10):
        self.dataframe = dataframe.reset_index(drop=True)
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        file_path = self.dataframe.iloc[idx]['file_path']
        label = self.dataframe.iloc[idx]['target']

        # Load and normalize
        speech, _ = librosa.load(file_path, sr=16000)
        if speech.ndim > 1:
            speech = speech.mean(axis=1)

        # Pad/truncate
        if len(speech) > self.max_length:
            speech = speech[:self.max_length]
        else:
            speech = np.pad(speech, (0, self.max_length - len(speech)))

        speech = speech.astype(np.float32)

        # Processor → [1, time]
        inputs = self.processor(
            speech,
            sampling_rate=16000,
            return_tensors="pt",
            padding=False,
            truncation=False
        )

        input_values = inputs["input_values"].squeeze(0)  # [time]

        return {
            "input_values": input_values,  # [time] - shape (160000,)
            "labels": torch.tensor(label, dtype=torch.long)
        }

#Train & Val
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['target'],
    random_state=42
)

train_dataset = AudioDataset(train_df, processor)
val_dataset   = AudioDataset(val_df,   processor)

# Test one sample
sample = train_dataset[0]
print("input_values shape (from dataset):", sample["input_values"].shape)  # Should be [160000]
print("labels:", sample["labels"])

# Move to GPU and add batch dimension for testing
sample_test = {k: v.to(model.device) for k, v in sample.items()}
input_values = sample_test["input_values"].unsqueeze(0)  # [1, 160000]
labels = sample_test["labels"].unsqueeze(0)  # [1]

model.eval()
with torch.no_grad():
    outputs = model(input_values=input_values, labels=labels)
    print("Logits shape:", outputs.logits.shape)  # [1, 4]
    print("Loss:", outputs.loss.item())

# Training arguments
# Training arguments
training_args = TrainingArguments(
    output_dir="./wav2vec2-finetuned",

    # # ================= TESTING MODE  =================
    # max_steps=10,             # FORCE STOP after 10 steps (overrides epochs)
    # eval_strategy="no",       # DISABLE EVAL (prevents waiting for validation)
    # save_strategy="no",       # DISABLE SAVE (saves disk I/O time)
    # # ============================================================

    ================= ORIGINAL MODE ============
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=2,
    ============================================================

    logging_strategy="steps",
    logging_steps=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=3e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    report_to=[],
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    disable_tqdm=False,
    log_level="info",
    logging_first_step=True,
)

class VerboseCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        print("Training started!\n")

    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"\n{'='*50}")
        print(f"END OF EPOCH {int(state.epoch)}")
        if state.log_history:
            latest = state.log_history[-1]
            print(f"   Train Loss: {latest.get('loss', 'N/A')}")
            print(f"   Eval Loss:  {latest.get('eval_loss', 'N/A')}")
            print(f"   Accuracy:   {latest.get('eval_accuracy', 'N/A')}")
        print(f"{'='*50}\n")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[VerboseCallback],
)

print("\n Testing forward pass...")
sample = train_dataset[0]
sample = {k: v.unsqueeze(0).to(model.device) for k, v in sample.items()}  # Add batch dim
with torch.no_grad():
    out = model(**sample)
print(f"✓ Forward pass OK → logits shape: {out.logits.shape}")

trainer.train()

final_path = "./wav2vec2-final"
model.save_pretrained(final_path)
processor.save_pretrained(final_path)
print(f"Final model saved to: {final_path}")


# Prediction

In [ ]:
import torch
import numpy as np
import pandas as pd
import librosa
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification
import os

processor = Wav2Vec2Processor.from_pretrained("./wav2vec2-final")
model = Wav2Vec2ForSequenceClassification.from_pretrained("./wav2vec2-final")

device = "cuda"
model = model.to(device)
model.eval()

In [ ]:
class TestAudioDataset(Dataset):
    def __init__(self, dataframe, processor, max_length=16000 * 10):
        self.dataframe = dataframe.reset_index(drop=True)
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        file_path = self.dataframe.iloc[idx]['file_path']
        speech, _ = librosa.load(file_path, sr=16000)
        if speech.ndim > 1:
            speech = speech.mean(axis=1)
        if len(speech) > self.max_length:
            speech = speech[:self.max_length]
        else:
            speech = np.pad(speech, (0, self.max_length - len(speech)))
        speech = speech.astype(np.float32)
        inputs = self.processor(speech, sampling_rate=16000, return_tensors="pt", padding=False)
        return {"input_values": inputs["input_values"].squeeze(0)}

test_df = pd.read_csv("/kaggle/input/comsys-hackathon-6-task-b-2-0/test_new_final.csv")
test_df["file_path"] = test_df["filename"].apply(lambda x: f"/kaggle/input/comsys-hackathon-6-task-b-2-0/ComsysTaskB/test/{x}")
test_dataset = TestAudioDataset(test_df, processor)

dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False)

all_preds = []
with torch.no_grad():
    for batch in dataloader:
        inputs = batch["input_values"].to(device)
        logits = model(input_values=inputs).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)

test_df["target"] = all_preds
test_df[["filename", "target"]].to_csv("submission.csv", index=False)
print("submission.csv saved!")

In [ ]:
import pandas as pd

sub = pd.read_csv('/kaggle/working/submission.csv')

# Change from a set {'filename', 'file_name'} to a dictionary
sub = sub.rename(columns={'filename': 'file_name'})

sub.to_csv("submission_c.csv" , index=False)